# Taxonomy inspection

Run `offline.taxonomy.run()`, eyeball each cluster's sample messages, assign an
intent name per cluster, then call `offline.taxonomy.write_intents()` to produce
the versioned `config/intents.json`.

In [ ]:
from config.settings import settings
from support_agent.offline import taxonomy

clusters = taxonomy.run(settings.artifact_dir, settings.processed_dir,22)
clusters

In [ ]:
for _, row in clusters.iterrows():
    print(f"cluster {row['cluster_id']} (n={row['size']})")
    for msg in row['sample_msgs']:
        print(f"  - {msg}")
    print()

In [ ]:
# Cluster -> intent, from the k=22 run over artifacts/v2.
# Several clusters share a name on purpose: write_intents() de-duplicates.
cluster_labels = {
    # -1 is not a cluster. See the note below on why this one is added by hand.
    -1: "fraud_or_unauthorized",

    0:  "pricing_or_promotion",
    1:  "contact_request",
    2:  "refund_status",
    3:  "item_damaged",
    4:  "other",                     # mid-thread fragments, mixed chatter
    5:  "no_action_needed",
    6:  "account_manage",
    7:  "account_access",
    8:  "no_response_received",
    9:  "delivery_late",
    10: "delivery_late",
    11: "device_or_content_issue",
    12: "service_feedback",
    13: "item_damaged",
    14: "device_or_content_issue",
    15: "item_wrong_or_missing",
    16: "other",                     # mid-thread fragments, exasperation
    17: "delivery_courier_issue",
    18: "delivery_not_received",
    19: "order_status",
    20: "delivery_wrong_location",
    21: "prime_membership",
}

descriptions = {
    "delivery_late": (
        "The order has not arrived yet and is past, or about to miss, its promised "
        "delivery date. Tracking still shows it in transit or processing."
    ),
    "delivery_not_received": (
        "Tracking says the package was delivered but the customer does not have it. "
        "Use this when the system claims completion and the customer disputes it."
    ),
    "delivery_wrong_location": (
        "The package went to the wrong address, wrong country, or wrong carrier "
        "facility, or the customer wants it rerouted to a different location."
    ),
    "delivery_courier_issue": (
        "A complaint about how the driver or courier handled the delivery: left in "
        "an unsafe place, signed for by someone else, damaged in handling, or stolen "
        "after being dropped off."
    ),
    "order_status": (
        "A question about an existing order itself - where it stands, how to cancel, "
        "modify, re-order, or start a return. Not about a delivery that has failed."
    ),
    "refund_status": (
        "A refund was requested, promised, or issued and the customer has not "
        "received the money, or the amount is wrong. Includes payment method problems."
    ),
    "pricing_or_promotion": (
        "A dispute about price, discount, delivery-charge eligibility, or a "
        "promotion not being honoured. The customer disagrees with what they were "
        "charged or offered, not with whether a refund arrived."
    ),
    "prime_membership": (
        "Questions or complaints about the Prime membership itself - its benefits, "
        "renewal, cancellation, or whether the service justifies its cost."
    ),
    "account_access": (
        "The customer cannot get into their account: locked out, password rejected, "
        "verification code not arriving, or the sign-in page will not let them in."
    ),
    "account_manage": (
        "Changing or administering an account the customer can already access: "
        "closing it, household members, addresses, registration details, settings."
    ),
    "item_damaged": (
        "The item or its packaging arrived damaged, broken, or counterfeit."
    ),
    "item_wrong_or_missing": (
        "The wrong item arrived, or part of an order is missing from a package that "
        "otherwise turned up."
    ),
    "device_or_content_issue": (
        "A device or digital service is not working: Echo, Alexa, Kindle, Fire TV, "
        "the app, Prime Video playback, or purchased digital content."
    ),
    "contact_request": (
        "The customer is asking how to reach a human - a phone number, live chat, or "
        "an escalation path. They want a channel, not an answer."
    ),
    "no_response_received": (
        "The customer completed a step they were asked to take, or is waiting on a "
        "promised email or callback that never came."
    ),
    "service_feedback": (
        "Feedback about the quality of support or service received, positive or "
        "negative, without a specific unresolved request attached."
    ),
    "no_action_needed": (
        "Thanks, acknowledgement, or confirmation that a problem is resolved. "
        "Nothing is being asked for."
    ),
    "fraud_or_unauthorized": (
        "The customer says something happened on their account that they did not do. "
        "Typical wording: 'items I never ordered arrived', 'charges I don't "
        "recognise', 'someone bought gift cards on my account', 'reviews I didn't "
        "write', 'my account has been hacked'. Choose this whenever the customer "
        "denies making the purchase or the charge - even when the message also "
        "mentions delivery, billing or refunds. It takes precedence over those."

    ),
    "other": (
        "Does not clearly match any other intent, or is a fragment of an ongoing "
        "conversation with no standalone meaning; catch-all."
    ),
}

taxonomy.write_intents(cluster_labels, descriptions, settings.intents_path, version="v2")

In [2]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

In [4]:
from support_agent.online.pipeline import handle

d = handle("smoke-1", "my order was supposed to arrive yesterday but it still hasn't")

print("intent    :", d.intent, f"({d.classify_confidence:.2f})")
print("top1_sim  :", round(d.top1_sim, 3))
print("action    :", d.action, "|", d.reason)
print("grounded  :", d.grounded_pair_ids)
print("reply     :", d.reply)

c:\Users\georg\Projects\support-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2747.58it/s]


intent    : delivery_late (0.90)
top1_sim  : 0.938
action    : auto | confident_and_grounded
grounded  : ['1564965_1564964', '2652393_2652392']
reply     : I'm sorry your order hasn't arrived. Check the carrier and tracking: https://t.co/Y5jpI9ys9c. Keep us posted.


In [5]:
tests = [
    ("t1", "my package says delivered but it never arrived"),
    ("t2", "I want a refund, I was charged twice"),
    ("t3", "¿dónde está mi paquete? llevo tres días esperando"),
    ("t4", "thanks!"),
    ("t5", "items I never ordered are arriving and someone is buying gift cards on my account"),
]
for tid, msg in tests:
    d = handle(tid, msg)
    print(f"{d.intent:24s} {d.classify_confidence:.2f} {d.top1_sim:.2f} {d.action:9s} {d.reason}")
    print(f"   {d.reply}\n")

delivery_not_received    0.99 0.94 auto      confident_and_grounded
   I'm sorry the delivery shows as completed but you haven't received it. Please check the Orders page for status and the Tracking page for details. If you still can't locate it, let me know and I can help investigate further.

refund_status            0.95 0.82 auto      below_escalation_threshold:refund_intent
   Hi! I see you were charged twice. First, check your Orders page to confirm there's no second order or pending authorization. If it looks correct, please open a refund request or contact us for assistance. Thanks!

delivery_late            0.95 0.77 auto      confident_and_grounded
   Hola, lamentamos la demora. Por favor revisa la sección Pedidos en tu app o web, o ve al enlace de seguimiento para ver el estado. Si no ves actualizaciones, contáctanos de nuevo.

no_action_needed         0.95 0.94 auto      confident_and_grounded
   You're welcome! If you need anything else, feel free to check your Orders page

In [3]:
import pandas as pd
from config.settings import settings

pairs = pd.read_parquet(settings.processed_dir / "pairs.parquet")
sample = pairs.sample(200, random_state=0)[["customer_tweet_id", "customer_msg"]]
sample.columns = ["tweet_id", "message"]
settings.labels_dir.mkdir(parents=True, exist_ok=True)
sample.to_csv(settings.labels_dir / "sample_200.csv", index=False)
print("wrote", len(sample), "rows")

wrote 200 rows


In [4]:
import httpx
from config.settings import settings
from support_agent.online.retrieve import retrieve
from support_agent.online.draft import _build_prompt, _SCHEMA

msg = "my package was marked delivered but the driver left it at the wrong building"
pairs = retrieve(msg)
prompt = _build_prompt(msg, pairs, "delivery_not_received", "en")
print("prompt chars:", len(prompt))

r = httpx.post("https://api.groq.com/openai/v1/chat/completions",
    headers={"Authorization": f"Bearer {settings.grok_api_key}"}, timeout=60,
    json={"model": settings.grok_model,
          "messages": [{"role": "user", "content": prompt}],
          "response_format": {"type": "json_schema",
                              "json_schema": {"name": "draft", "schema": _SCHEMA}}})
print("status:", r.status_code)
d = r.json()
print("finish_reason:", d.get("choices", [{}])[0].get("finish_reason"))
print("usage:", d.get("usage"))
print("content:", repr(d.get("choices", [{}])[0].get("message", {}).get("content"))[:400])
print("raw error:", r.text[:400] if r.status_code != 200 else "")

c:\Users\georg\Projects\support-agent\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2045.78it/s]


prompt chars: 2836
status: 200
finish_reason: stop
usage: {'queue_time': 0.003310788, 'prompt_tokens': 853, 'prompt_time': 0.050185786, 'completion_tokens': 395, 'completion_time': 0.446541791, 'total_tokens': 1248, 'total_time': 0.496727577, 'completion_tokens_details': {'reasoning_tokens': 300}}
content: '{"reply":"Sorry to hear that. Please check your Orders or tracking page to confirm the delivery address. If it’s still missing, contact your carrier to confirm where it was left. If it can’t be located, let us know so we can help investigate.","grounded_pair_ids":["1077739_1077741","2286555_2286554","1511992_1511991"]}'
raw error: 
